# 20 — paper, DOCX edition

Builds `docs/paper/fedopd_paper.docx`: the single-column, short-sentence edition of the
conference paper. At most 4,500 words of text. All eleven tables are generated directly from
the stored result csvs and all fourteen report figures are embedded, so no number is typed
into a table by hand. The last cell reopens the saved file and pins the headline prose numbers
to csv recomputations — a changed csv stops the notebook instead of shipping stale prose.

In [1]:
# Goal: load every result csv the document cites and shape the eleven display tables.
# Each table becomes (display strings, bold mask), so the builder cells never touch a number.

from pathlib import Path   # stdlib path objects: keeps the ../data file locations readable
import numpy as np         # numerical library: the argmin/rounding arithmetic behind the bolds
import pandas as pd        # dataframes: every stored result csv is read and pivoted with it

ROOT = Path("..")                     # the notebook executes from src/, the repo root is one up
PROC = ROOT / "data" / "processed"    # result csvs written by the analysis notebooks
EVALS = ROOT / "output" / "evals"     # small evaluation csvs written by the report notebooks
FIGS = ROOT / "output" / "figures"    # the 14 report pngs, embedded unchanged

# Treatment-function names as read from the RTT reference (2026 schema, "Service" suffix
# dropped), the same mapping recorded in fedopd_paper.tex. Codes key every per-specialty csv.
NAMES = {
    "C_100": "General Surgery", "C_101": "Urology", "C_110": "Trauma & Orthopaedics",
    "C_120": "Ear, Nose & Throat", "C_130": "Ophthalmology", "C_140": "Oral Surgery",
    "C_150": "Neurosurgery", "C_160": "Plastic Surgery", "C_170": "Cardiothoracic Surgery",
    "C_300": "General Internal Medicine", "C_301": "Gastroenterology", "C_320": "Cardiology",
    "C_330": "Dermatology", "C_340": "Respiratory Medicine", "C_400": "Neurology",
    "C_410": "Rheumatology", "C_430": "Elderly Medicine", "C_502": "Gynaecology",
    "X01": "Other (pre-2021 schema)", "X02": "Other: Medical", "X03": "Other: Mental Health",
    "X04": "Other: Paediatric", "X05": "Other: Surgical", "X06": "Other: Other",
}


def fmt1(v):
    # every MAE/score cell shares one convention: 1 decimal, thousands comma, "-" if absent
    return "-" if pd.isna(v) else f"{v:,.1f}"


def fmt0(v):
    # national-grain errors are thousands of patients: whole numbers read better there
    return "-" if pd.isna(v) else f"{v:,.0f}"


def bold_mask(num, trained, pers):
    # The bold rule of the three per-specialty tables: mark the single best trained model on
    # the UNROUNDED value where it beats persistence, else mark persistence itself.
    bold = pd.DataFrame(False, index=num.index, columns=num.columns)
    for code, row in num.iterrows():
        if row[trained].isna().all():   # X01 has zero validation rows: nothing to mark
            continue
        best = row[trained].astype(float).idxmin()
        bold.loc[code, best if row[best] < row[pers] else pers] = True
    return bold


TABLES = {}   # table key -> (display DataFrame of strings, bold mask or None)

# ---- Table 1: national era summary ----------------------------------------------------
era = pd.read_csv(EVALS / "eval_national_pre_post.csv")
TABLES["era"] = (pd.DataFrame({
    "era": ["pre-Covid", "post-Covid"],
    "months": era["n_months"].astype(str),
    "mean list": [f"{v:,.0f}" for v in era["mean_waiting"]],
    ">18w share": [f"{v * 100:.1f}%" for v in era["mean_pct_gt18"]],
    ">52w share": [f"{v * 100:.2f}%" for v in era["mean_pct_gt52"]],
}), None)

# ---- Table 2: validation MAE by horizon and grain --------------------------------------
hz = pd.read_csv(PROC / "val_horizons_c110.csv").pivot_table(
    index="h", columns=["model", "regime"], values="mae")
hzc = pd.read_csv(PROC / "val_horizons_coarse_c110.csv").pivot_table(
    index=["grain", "h"], columns=["model", "regime"], values="mae")
H = [1, 3, 6, 12]
HZ_ROWS = [   # grain label, model label, source pivot, (model, regime) column, formatter;
              # challengers shown full-history (post-only is worse everywhere, per the csv)
    ("provider", "persistence", hz, ("persistence", "-"), fmt1),
    ("provider", "drift", hz, ("drift", "full"), fmt1),
    ("provider", "GBM+flows", hz, ("gbm-delta+flows", "full"), fmt1),
    ("provider", "seasonal", hz, ("seasonal", "-"), fmt1),
    ("ICB", "persistence", hzc.loc["icb"], ("persistence", "-"), fmt1),
    ("ICB", "drift", hzc.loc["icb"], ("drift", "full"), fmt1),
    ("national", "persistence", hzc.loc["national"], ("persistence", "-"), fmt0),
    ("national", "drift", hzc.loc["national"], ("drift", "full"), fmt0),
]
TABLES["horizons"] = (pd.DataFrame(
    # .get(h): seasonal stores no h=12 row (the 12-back lag falls outside validation), so
    # that one cell is missing and renders as "-"
    [[g, m] + [f(src[col].get(h)) for h in H] for g, m, src, col, f in HZ_ROWS],
    columns=["grain", "model", "h=1", "h=3", "h=6", "h=12"]), None)

# ---- Table 3: persistence MAE per validation origin month ------------------------------
mo = pd.read_csv(EVALS / "eval_c110_persistence_mae_by_month_h1.csv")
TABLES["monthly"] = (pd.DataFrame({
    "origin month": mo["month"].str[:7],   # "2024-04-01" -> "2024-04"
    "n": mo["n"].astype(str),
    "MAE": [fmt1(v) for v in mo["mae_naive"]],
}), None)

# ---- Table 4: C_110 linear FL summary ---------------------------------------------------
flp = pd.read_csv(PROC / "fl_summary_c110.csv").pivot_table(
    index="model", columns="regime", values=["mae", "rmse"])
FL_ROWS = [   # display name, csv model key, regime key holding its full-history scores;
              # persistence is stored regime-less ("-"), so its post-Covid cell is empty
    ("naive persistence", "naive persistence", "-"),
    ("pooled (centralised)", "pooled", "full"),
    ("FedAvg, delta space", "fedavg-delta", "full"),
    ("FedAvg, delta + local intercept", "fedavg-delta+local-a", "full"),
    ("local-only (level space)", "local-only", "full"),
    ("FedAvg, level space", "fedavg", "full"),
]


def mae_rmse(model, regime):
    # one cell = "MAE / RMSE" for one model in one regime
    v = flp.loc[model, ("mae", regime)]
    return "-" if pd.isna(v) else f"{v:.1f} / {flp.loc[model, ('rmse', regime)]:.1f}"


TABLES["fl"] = (pd.DataFrame({
    "model": [d for d, m, r in FL_ROWS],
    "full history (MAE / RMSE)": [mae_rmse(m, r) for d, m, r in FL_ROWS],
    "post-Covid (MAE / RMSE)": [mae_rmse(m, "post") for d, m, r in FL_ROWS],
}), None)

# ---- Table 5: FL scale-up per specialty (linear delta families) -------------------------
sc = pd.read_csv(PROC / "fl_scaleup_by_specialty.csv")   # csv is sorted by n_val descending
SC_COLS = {"persistence": "persistence_mae", "local": "local_full_mae",
           "pooled": "pooled_full_mae", "FedAvg-Δ": "fedavg_delta_full_mae",
           "FedAvg-Δ+local-a": "fedavg_delta_local_a_full_mae"}
sc_num = sc.set_index("tfunc_code")[list(SC_COLS.values())]
sc_num.columns = list(SC_COLS)
sc_disp = pd.DataFrame(index=sc_num.index)
sc_disp["code"] = sc_num.index
sc_disp["treatment function"] = [NAMES[c] for c in sc_num.index]
sc_disp["n_val"] = [f"{v:,}" for v in sc.set_index("tfunc_code")["n_val"]]
sc_disp["clients"] = sc.set_index("tfunc_code")["n_clients"].astype(str)
for c in SC_COLS:
    sc_disp[c] = [fmt1(v) for v in sc_num[c]]
TABLES["scaleup"] = (sc_disp, bold_mask(sc_num, [c for c in SC_COLS if c != "persistence"],
                                        "persistence"))

# ---- Table 6: neural vs linear per specialty, full regime -------------------------------
nn = pd.read_csv(PROC / "fl_neural_summary.csv")
pers_nn = nn[nn["model"] == "persistence"].set_index("tfunc_code")
fullp = nn[nn["regime"] == "full"].pivot(index="tfunc_code", columns="model", values="mae")
NN_COLS = {"linear pooled": "linear_pooled", "linear FedAvg": "linear_fedavg",
           "neural pooled": "neural_pooled", "neural FedAvg": "neural_fedavg"}
nn_num = fullp[list(NN_COLS.values())].copy()
nn_num.columns = list(NN_COLS)
nn_num.insert(0, "persistence", pers_nn["mae"])
nn_disp = pd.DataFrame(index=nn_num.index)
nn_disp["code"] = nn_num.index
nn_disp["treatment function"] = [NAMES[c] for c in nn_num.index]
nn_disp["n"] = [f"{v:,}" for v in pers_nn.loc[nn_num.index, "n"]]
for c in nn_num.columns:
    nn_disp[c] = [fmt1(v) for v in nn_num[c]]
TABLES["neural"] = (nn_disp, bold_mask(nn_num, list(NN_COLS), "persistence"))

# ---- Table 7: declared multi-seed check at C_110 ----------------------------------------
ms = pd.read_csv(PROC / "fl_neural_c110_multiseed.csv")
msp = ms.pivot_table(index=["regime", "model"], columns="seed", values="mae")
ms_rows = []
for reg, mod in [("full", "neural_pooled"), ("full", "neural_fedavg"),
                 ("post", "neural_pooled"), ("post", "neural_fedavg")]:
    v = msp.loc[(reg, mod)]
    ms_rows.append([reg, mod.replace("_", " ")] + [f"{v[s]:.1f}" for s in (42, 123, 999)]
                   + [f"{v.mean():.1f} ± {v.std(ddof=1):.1f}"])   # 3 seeds = a sample: ddof=1
TABLES["multiseed"] = (pd.DataFrame(
    ms_rows, columns=["regime", "model", "seed 42", "seed 123", "seed 999", "mean ± sd"]), None)

# ---- Table 8: frozen agent rules --------------------------------------------------------
ag = pd.read_csv(PROC / "agent_frozen_summary.csv")
TABLES["agent"] = (pd.DataFrame({
    "rule": ag["rule"], "decision": ag["decision"],
    "n": [f"{v:,}" for v in ag["n"]],
    "share of 2,337": [f"{v * 100:.1f}%" for v in ag["share_of_2337"]],
    "median pressure": [f"{v:.3f}" for v in ag["median_pressure"]],
    "median 6m change": [f"{v:+,.1f}" for v in ag["median_realized_change"]],
}), None)

# ---- Table 9: v2 stress test ------------------------------------------------------------
st = pd.read_csv(EVALS / "eval_c110_agent_stress_h6.csv")
TABLES["stress"] = (pd.DataFrame({
    "window": st["window"], "class": st["decision_v2"],
    "n": st["n"].astype(str),
    "median pressure": [f"{v:.3f}" for v in st["median_pressure"]],
    "median 6m change": [f"{v:+,.1f}" for v in st["median_realized_change"]],
}), None)

# ---- Table 10: sealed year at the anchor ------------------------------------------------
se = pd.read_csv(PROC / "sealed_year_eval.csv")
c110 = se[se["tfunc_code"] == "C_110"].set_index(["model", "train_cut"])
SE_ROWS = [
    ("persistence", "-", "persistence"),
    ("neural_pooled", "final", "neural pooled (final, seed 42)"),
    ("neural_fedavg", "final", "neural FedAvg (final, seed 42)"),
    ("linear_pooled", "final", "linear pooled (final)"),
    ("linear_fedavg", "final", "linear FedAvg (final)"),
    ("linear_pooled", "val_frozen", "linear pooled (val-frozen)"),
    ("linear_fedavg", "val_frozen", "linear FedAvg (val-frozen)"),
]
TABLES["sealed_anchor"] = (pd.DataFrame({
    "model": [d for m, c, d in SE_ROWS],
    "sealed FY 2025-26": [fmt1(c110.loc[(m, c), "mae"]) for m, c, d in SE_ROWS],
    "validation FY 2024-25": [fmt1(c110.loc[(m, c), "val_mae_full"]) for m, c, d in SE_ROWS],
}), None)

# ---- Table 11: sealed year, complete one-shot per specialty -----------------------------
fin = se[(se["train_cut"] != "val_frozen") & (se["model"] != "persistence")].pivot(
    index="tfunc_code", columns="model", values="mae")
vf = se[se["train_cut"] == "val_frozen"].pivot(index="tfunc_code", columns="model", values="mae")
sealedp = se[se["model"] == "persistence"].set_index("tfunc_code")
sl_num = pd.DataFrame({
    "persistence": sealedp["mae"],
    "linear FedAvg": fin["linear_fedavg"], "linear pooled": fin["linear_pooled"],
    "neural FedAvg": fin["neural_fedavg"], "neural pooled": fin["neural_pooled"],
    "val-frozen linear FedAvg": vf["linear_fedavg"],
})
sl_disp = pd.DataFrame(index=sl_num.index)
sl_disp["code"] = sl_num.index
sl_disp["treatment function"] = [NAMES[c] for c in sl_num.index]
sl_disp["n"] = [f"{v:,}" for v in sealedp.loc[sl_num.index, "n"]]
for c in sl_num.columns:
    sl_disp[c] = [fmt1(v) for v in sl_num[c]]
TABLES["sealed_full"] = (sl_disp, bold_mask(
    sl_num, [c for c in sl_num.columns if c != "persistence"], "persistence"))

# ---- Pins: headline numbers the prose quotes, recomputed here from the csvs -------------
# The verification cell asserts each formatted pin appears verbatim in the document text,
# so a changed result csv stops the notebook instead of shipping stale prose.
drift_abs = ((sealedp["mae"] / sealedp["val_mae_full"] - 1) * 100).abs().median()
all_trained = pd.concat([fin, vf.add_prefix("vf_")], axis=1)
margin_max = ((1 - all_trained.div(sealedp["mae"], axis=0)) * 100).max().max()
PINS = [
    f"{flp.loc['naive persistence', ('mae', '-')]:.1f}",   # 153.5: validation persistence MAE
    f"{flp.loc['fedavg', ('mae', 'full')]:.1f}",           # 186.5: level-space FedAvg failure
    f"{flp.loc['fedavg-delta', ('mae', 'full')]:.1f}",     # 154.0: delta-space FedAvg
    f"{flp.loc['pooled', ('mae', 'full')]:.1f}",           # 153.8: centralised pooling
    f"{fullp.loc['C_110', 'neural_pooled']:.1f}",          # 150.6: anchor neural pooled
    f"{fullp.loc['C_110', 'neural_fedavg']:.1f}",          # 152.4: anchor neural FedAvg
    f"{pers_nn.loc['C_110', 'mae']:.1f}",                  # 154.1: anchor eligible persistence
    f"{sealedp.loc['C_110', 'mae']:.1f}",                  # 151.5: sealed anchor persistence
    f"{fin.loc['C_110', 'neural_pooled']:.1f}",            # 151.6: sealed anchor neural pooled
    f"{drift_abs:.1f}",                                    # 9.3: median absolute drift, percent
    f"{margin_max:.1f}",                                   # 4.8: largest model margin, percent
]
print("tables built:", len(TABLES), "| pins:", ", ".join(PINS))

tables built: 11 | pins: 153.5, 186.5, 154.0, 153.8, 150.6, 152.4, 154.1, 151.5, 151.6, 9.3, 4.8


In [2]:
# Goal: the document as data. PLAN is the full paper in reading order: headings, short-sentence
# prose, and the slots where each csv-built table and figure png lands. The builder cell renders
# it without editing a word. Every number in the prose is pinned to a csv recomputation (cell 1)
# or transcribed from a stored notebook print named in docs/WORKLOG.md.

TITLE = ["Federation Is Free, Foresight Is Not:",
         "Federated Learning, Frozen Rules, and a Sealed Year of NHS Waiting Lists"]
AUTHOR = ["Nilesh Sarkar   |   nileshsarkar.cs@gmail.com",
          "Code, data pipeline and all stored outputs: https://github.com/nileshsarkar-ai/fedOPD"]

PLAN = [
    ("h1", "Abstract"),
    ("p", "The English NHS elective waiting list grew from a monthly mean of 4.28 million "
          "pathways before Covid-19 to 6.57 million after. The share waiting over 52 weeks rose "
          "81-fold, from 0.05% to 4.11%. Allocation decisions need forecasts per provider and "
          "specialty. Patient-level data cannot be centralised freely. We built fedOPD to measure "
          "what is actually achievable. The pipeline audits 96 months of public "
          "Referral-to-Treatment (RTT) and KH03 bed data. The panel holds 343,441 "
          "provider-specialty-month rows from 651 providers. We evaluate demand forecasting, "
          "federated learning (FL) and rule-based allocation agents. The protocol is "
          "pre-registered. The final year (FY 2025-26) was sealed before any model was fit. It "
          "was read exactly once."),
    ("p", "On validation, naive persistence is unbeaten by nine model families at every spatial "
          "grain and horizon. Its MAE is 153.5 over 1,407 rows at the largest specialty. "
          "Level-space FedAvg fails on non-IID clients (MAE 186.5). Reformulating clients in "
          "delta space repairs federation completely: federated MAE 154.0 versus 153.8 for "
          "centralised pooling, with a median gap of -0.08 MAE across the 23 scored specialties. "
          "Collaboration without raw data sharing costs nothing. GRU clients produced the anchor "
          "specialty's only sub-persistence validation scores (pooled 150.6, federated 152.4, "
          "versus 154.1). Three declared seeds confirmed the result. Yet on the sealed year, "
          "persistence retook the lead by 0.1 MAE. Models still won at 14 to 15 of 23 "
          "specialties. Year-over-year drift in persistence error (median absolute change 9.3%) "
          "dwarfs the largest model margin measured anywhere (4.8%). The deployable product is "
          "calibrated uncertainty (conformal bands, 87.6-89.6% empirical coverage) plus frozen "
          "decision rules. Federation is a free privacy layer, not a better point forecast."),
    ("p", "Keywords: federated learning; NHS waiting lists; demand forecasting; conformal "
          "prediction; agentic decision support; healthcare resource allocation."),

    ("h1", "1. Introduction"),
    ("p", "Elective care in England runs on Referral-to-Treatment (RTT) pathways. Each provider "
          "reports each specialty monthly [16]. The post-pandemic backlog turned these statistics "
          "into a planning instrument. Integrated care boards (ICBs) decide where to expand "
          "capacity, when to redistribute demand, and when to buy independent-sector activity. "
          "These decisions assume the near future of each waiting list is predictable. They also "
          "assume the data needed to predict it can be pooled. Both assumptions deserve "
          "measurement rather than faith."),
    ("p", "Two obstacles stand out. First, hospital operations are a natural home for federated "
          "learning [1]. Organisations that cannot share records could still train a shared "
          "model. But hospitals are strongly non-IID clients. A teaching trust and a specialist "
          "centre hold lists that differ by orders of magnitude. That is the regime where naive "
          "federated averaging degrades [2], [3]. Second, simple benchmarks are hard to beat on "
          "noisy operational series [7], [8]. A federated model that loses to “next month "
          "equals this month” is an expensive way to be wrong."),
    ("p", "This paper reports fedOPD, an end-to-end study on eight years of public NHS data, run "
          "with the discipline of a prospective trial. Our contributions: (i) a fully audited "
          "96-month panel joining RTT waiting lists to KH03 bed capacity, with every documented "
          "number reproduced by asserts inside the notebooks that compute it. (ii) A baseline "
          "closure result: persistence is unbeaten on MAE by nine model families at every grain "
          "and horizon on a held-out validation year. (iii) A federation result: level-space "
          "FedAvg fails on non-IID hospitals, but delta-space reformulation makes federated "
          "clients match centralised pooling. (iv) Frozen, ratified allocation rules whose "
          "simulated decisions separate real populations, plus a structural finding that "
          "in-ICB-only redistribution is geographically starved. (v) A one-shot sealed-year "
          "evaluation, pre-registered before execution. Complete run-result tables appear in "
          "Section 5."),

    ("h1", "2. Related Work"),
    ("p", "McMahan et al. introduced federated averaging [1]. Zhao et al. documented its "
          "degradation under non-IID clients [2]. FedProx is an algorithmic remedy [3]. Kairouz "
          "et al. survey the field [4]. Healthcare is a canonical FL motivation. Rieke et al. "
          "argue for federation in digital health [5]. Sheller et al. demonstrated "
          "multi-institutional training without data sharing [6]. Most health-FL studies compare "
          "learned models against each other. They rarely benchmark against the operational "
          "naive forecasts practitioners already use."),
    ("p", "In forecasting, the M4 competition showed that naive benchmarks are frequently "
          "competitive on aggregate operational series [7]. Standard texts say the same [8]. "
          "Queueing views of hospital waiting lists date back to Worthington [9]. Policy "
          "analyses caution that system dynamics dominate marginal predictive gains [10]. For "
          "uncertainty, conformal prediction gives distribution-free coverage guarantees [11], "
          "[12], [13]. Our neural clients use standard parts: gated recurrent units [14] trained "
          "with Adam [15]."),

    ("h1", "3. Research Gap"),
    ("p", "Three gaps motivate the study. First, FL for health operations is rarely benchmarked "
          "against naive forecasts on public data anyone can audit. Second, reported gains are "
          "seldom tested prospectively. A sealed, pre-registered final year is standard in "
          "clinical research. It is almost absent from FL forecasting papers. Third, allocation "
          "(“agentic”) rules are usually decoupled from the measured uncertainty of "
          "the forecasts that trigger them. fedOPD addresses all three on one pipeline."),

    ("h1", "4. Methodology"),
    ("h2", "4.1 Data and panel construction"),
    ("p", "We ingest all 96 monthly RTT “full extract” CSVs, April 2018 to March 2026 "
          "[16]. A schema break in April 2021 is handled by name-derived band mapping. We add "
          "quarterly KH03 bed availability and occupancy [17], and the ODS trust reference. The "
          "canonical panel has 343,441 provider-specialty-month rows and 24 columns: list size, "
          ">18-week and >52-week counts and shares, trust metadata, ICB code, and "
          "general-and-acute occupancy. The NHS aggregate row (C_999) is excluded everywhere. "
          "Including it silently doubles national totals. 651 providers appear. The 193 public "
          "trusts hold 94.5% of the list. Fig. 1 shows the national series. Fig. 2 shows the "
          "long-wait shares. Table 1 gives the era summary."),
    ("fig", ("fig_national_trend.png",
             "Fig. 1. National RTT waiting list, April 2018 to March 2026 (96 months, all "
             "providers and specialties). The dashed line is the Covid regime break (March "
             "2020). The shaded band is the sealed test year.", 3.1)),
    ("fig", ("fig_national_longwait_shares.png",
             "Fig. 2. Monthly >18-week and >52-week shares of the national list (denominator: "
             "that month's total incomplete pathways).", 3.1)),
    ("table", ("era", "Table 1. National era summary: means of monthly values "
                      "(eval_national_pre_post.csv; 23 pre-Covid and 73 post-Covid months).")),
    ("h2", "4.2 Evaluation protocol, declared once"),
    ("p", "The final fiscal year FY 2025-26 was sealed before any model was fit. The development "
          "cut (month < 2025-04-01) is applied before target construction. Sealed targets are "
          "therefore unreachable by construction. Every modelling notebook carries a sentinel "
          "assert on origin months. FY 2024-25 is the validation year. One seed (42) is "
          "declared. The Covid break (March 2020) defines a “post-only” training "
          "regime evaluated alongside full history. Nothing was ever adjusted after seeing a "
          "metric. Not thresholds, not splits, not horizons."),
    ("h2", "4.3 Forecasting layer and uncertainty"),
    ("p", "The baseline is persistence: next month equals this month. Challengers on identical "
          "validation rows include drift, seasonal-naive, gradient-boosted trees over lag and "
          "occupancy features, and referral-flow covariates. Uncertainty is a split-conformal "
          "band [11]. Calibration residuals come from FY 2023-24. The band is the persistence "
          "forecast plus or minus the 90th percentile of those residuals. Empirical validation "
          "coverage is 87.6-89.6% against a 90% target. The band grows from ±298 patients "
          "at h=1 to ±1,714 at h=12."),
    ("h2", "4.4 Federated clients"),
    ("p", "Clients are providers within one specialty. The pool is the 149 public providers "
          "flagged fl_client: at least 48 months of history and reporting in the final month. "
          "We compare four information regimes: local-only, centralised pooling, and federated "
          "averaging in two model spaces. The level-space client regresses the next level on "
          "the current level and the latest change: ŷ(t+1) = β0 + β1·y(t) + "
          "β2·(y(t) - y(t-1)). The delta-space client predicts the change instead: "
          "Δŷ(t+1) = a + b·(y(t) - y(t-1)), then ŷ(t+1) = y(t) + "
          "Δŷ(t+1). Both are exact least squares per client. Federation is one "
          "sample-weighted coefficient average. Only two or three coefficients and a count ever "
          "leave a client."),
    ("p", "The neural client is a single-layer GRU [14] (input 4, hidden 32) with a linear "
          "head. It reads six-month windows of log(1+y), Δy/100, and the two wait shares. "
          "All transforms are statistics-free. No scaler is fitted anywhere. It predicts "
          "Δy/100. Federation is classic FedAvg [1]: 10 rounds, 5 local full-batch Adam "
          "[15] steps per round (learning rate 0.001), then the weighted average of every "
          "tensor. The pooled reference gets the identical budget of 50 epochs. All "
          "hyperparameters were declared a priori. There are no sweeps and no early stopping. "
          "Every model, persistence included, is rescored on identical window-eligible rows. At "
          "the anchor specialty C_110 that is 1,401 of the standard 1,407 validation rows. No "
          "comparison ever mixes denominators."),
    ("h2", "4.5 Agent rules"),
    ("p", "A provider-month pressure score is a weighted within-month percentile rank: p = "
          "0.2·rank(>18w share) + 0.6·rank(>52w share) + 0.2·rank(occupancy). "
          "The >52-week share carries the largest weight. It is the component that moved "
          "81-fold after Covid. Rule set v1 combines pressure thresholds (0.70/0.40) with the "
          "6-month conformal band (growth-risk 1.20x, stability 1.05x) and occupancy slack "
          "(0.80). Rule set v2 classifies within-ICB senders and receivers by ICB-internal "
          "ranks (sender at or above 0.80, receiver at or below 0.30). The constants were "
          "ratified and frozen on 2026-07-30. The freeze notebook re-derives all 2,337 "
          "simulated decisions from stored components and asserts bit-for-bit equality. A "
          "minimum receiver group emerges arithmetically. A rank at or below 0.30 needs an ICB "
          "of at least ceil(1/0.30) = 4 providers."),
    ("h2", "4.6 The sealed-year one-shot"),
    ("p", "The final notebook is pre-registered in its own header before execution. "
          "Full-history regime. Linear clients in two cuts: val-frozen (the exact validated "
          "fits) and final (refit on targets through March 2025). Neural clients final-cut "
          "only, at seed 42. Test rows are windows whose origin lies in FY 2025-26, with "
          "targets May 2025 to March 2026. It ran exactly once. The train/test split is "
          "asserted on target months. The boundary window (origin March 2025) belongs to "
          "neither set."),

    ("h1", "5. Results and Discussion"),
    ("p", "All numbers below are transcribed from the stored result csvs written by the "
          "notebooks (data/processed/, tracked in the repository). The tables in this document "
          "are generated directly from those csvs by the build notebook. All nineteen analysis "
          "notebooks (01-19) execute headless end-to-end with zero stored errors."),
    ("h2", "5.1 The random walk is the forecast to beat, and it wins"),
    ("p", "On validation, persistence achieves MAE 153.5 (RMSE 330.1) over 1,407 provider-rows "
          "at C_110 (Trauma & Orthopaedics). No challenger beats it at any horizon or grain. "
          "Table 2 gives the full horizon grid. Fig. 3 shows the same picture graphically. "
          "Heterogeneity is large. Monthly persistence MAE ranges 123.3 to 221.9 across the "
          "validation year (Table 3). The per-provider distribution is heavily right-skewed "
          "(Fig. 4; median 113). The target itself is volatile. That foreshadows the "
          "sealed-year result."),
    ("table", ("horizons", "Table 2. Validation MAE by horizon and grain (val_horizons_c110.csv, "
                           "val_horizons_coarse_c110.csv). Provider grain: the 131-provider "
                           "cohort. Challengers shown for the full-history regime; post-only is "
                           "worse everywhere.")),
    ("table", ("monthly", "Table 3. Persistence MAE per validation origin month, C_110, h=1 "
                          "(eval_c110_persistence_mae_by_month_h1.csv; n = provider rows scored "
                          "that month).")),
    ("fig", ("fig_c110_horizon_mae_by_grain.png",
             "Fig. 3. Persistence vs challengers by horizon at provider, ICB and national grain "
             "(validation). The y-scales differ by about 30x between grains. Persistence wins "
             "in every panel.", 5.5)),
    ("fig", ("fig_c110_provider_mae_h1.png",
             "Fig. 4. Per-provider 1-month persistence MAE, C_110, FY 2024-25 (128 providers; "
             "median 113).", 3.1)),
    ("h2", "5.2 Federation is free in delta space"),
    ("p", "Table 4 and Fig. 5 give the federation story at the anchor. Level-space FedAvg "
          "degrades badly on non-IID hospitals: MAE 186.5 on full history, 228.8 post-only. "
          "Averaging intercepts across providers of wildly different scale is the textbook "
          "failure [2]. The delta reformulation removes the scale. Federated MAE is 154.0 "
          "versus 153.8 pooled. Per client, federated coefficients beat local fits at 41.4% of "
          "128 clients (median per-client MAE: local 113, federated 134, naive 113). Table 5 "
          "scales the comparison to every specialty in the pool (26,504 validation rows). The "
          "federated-minus-pooled gap has median -0.08 MAE (worst +1.27, best -4.61). A delta "
          "model edges persistence at 12 of the 23 scored specialties. The largest margin is "
          "C_120, Ear Nose & Throat: 165.1 versus 171.6. The baseline closure is therefore "
          "anchor-specific. But margins top out at 6.5 MAE."),
    ("table", ("fl", "Table 4. Validation MAE / RMSE at C_110 on the identical 1,407 rows "
                     "(fl_summary_c110.csv).")),
    ("fig", ("fig_c110_fl_models_h1.png",
             "Fig. 5. Level-space federation fails; delta-space federation matches centralised "
             "pooling (validation, both regimes, same 1,407 rows).", 3.1)),
    ("table", ("scaleup", "Table 5. FL scale-up: full-history validation MAE per specialty "
                          "(fl_scaleup_by_specialty.csv; 24 specialties, 149 public fl_client "
                          "providers, 26,504 validation rows). Local and federated models are "
                          "delta-space. Bold = best trained model where it also beats "
                          "persistence. X01 has zero validation rows in this pool.")),
    ("h2", "5.3 Neural clients: a coin flip, except at the anchor"),
    ("p", "Table 6 compares the linear and neural families per specialty on identical "
          "window-eligible rows. The federated GRU beats federated linear at 11 of 23 "
          "specialties (median gap +0.15 MAE). Across 2,441 client-specialty pairs it wins "
          "55.2%. That is a coin flip (Figs. 6, 7, 8). Where signal exists, federation costs "
          "neural models more than linear ones. At C_120: neural pooled 163.9, neural FedAvg "
          "168.6, linear FedAvg 165.5. Averaging nonlinear weights loses more than averaging "
          "two coefficients. The exception is the anchor. Neural pooled scores 150.6 and "
          "neural FedAvg 152.4 versus persistence 154.1 on 1,401 identical rows. Every "
          "sub-persistence score recorded at the anchor, in either regime, is neural. No "
          "linear variant gets under 154.1. A declared three-seed check confirms the result "
          "(Table 7): all six full-regime scores beat persistence. Dataset size does not "
          "decide the winner. The size-gap correlation is r = 0.128 (Fig. 9). The four "
          "largest relative neural wins are the administrative X-codes, exactly the "
          "specialties with no pre-Covid history. The delta signal is thin everywhere. The "
          "highest delta-space R² anywhere is +0.075."),
    ("table", ("neural", "Table 6. Neural vs linear clients: full-regime validation MAE per "
                         "specialty on identical window-eligible rows (fl_neural_summary.csv). "
                         "Bold = best trained model where it also beats persistence.")),
    ("table", ("multiseed", "Table 7. Declared multi-seed check of the C_110 neural result, "
                            "n=1,401, persistence 154.1 (fl_neural_c110_multiseed.csv). Verdict "
                            "rule fixed before running: confirmed iff all three full-regime "
                            "seeds beat persistence.")),
    ("fig", ("fig_c110_fl_linear_vs_neural_h1.png",
             "Fig. 6. Linear vs neural families at C_110 on identical eligible rows "
             "(validation, n=1,401).", 3.1)),
    ("fig", ("fl_neural_vs_linear.png",
             "Fig. 7. Neural vs linear federated MAE, one point per specialty, both regimes "
             "(identical validation rows per point; written by the neural notebook). Points "
             "below the diagonal favour the neural model.", 5.5)),
    ("fig", ("fig_fl_neural_gap_by_specialty_h1.png",
             "Fig. 8. Neural minus linear FedAvg MAE per specialty, both regimes (below zero "
             "favours neural): 11 of 23 below zero (full regime), 12 of 23 (post). A coin "
             "flip, not a takeover.", 3.1)),
    ("fig", ("fig_fl_neural_gap_vs_size_h1.png",
             "Fig. 9. Relative neural-linear FedAvg gap vs specialty size (Pearson r = 0.128, "
             "23 specialties). Size does not decide the winner. The largest neural wins are "
             "the no-pre-Covid X-codes.", 3.1)),
    ("h2", "5.4 Covid regime: keep the history"),
    ("p", "Once models are scale-free, discarding pre-Covid history hurts. Full-history "
          "training beats post-only at 17 of 18 clinical specialties for linear FedAvg, and at "
          "15 of 18 for neural FedAvg. Each family is compared within its own fixed validation "
          "rows (Fig. 10). The five X-codes sit exactly at zero. They have no pre-Covid rows "
          "to drop. The largest penalty is C_502, Gynaecology: +3.3 MAE linear, +2.4 neural. "
          "The regime break that dominates level models barely matters in delta space."),
    ("fig", ("fig_fl_covid_regime_effect_h1.png",
             "Fig. 10. Cost of training on post-Covid data only (post minus full MAE), per "
             "specialty and family.", 3.1)),
    ("h2", "5.5 Agent rules: clean signals, starved geography"),
    ("p", "Table 8 summarises the frozen rules over 2,337 provider-months (April 2023 to "
          "September 2024, 132 providers). Table 9 gives the three-window stress test. Fig. 11 "
          "shows the v2 classes against what happened six months later. The frozen v2 rule "
          "flags 457 sender months and 105 receiver months. Their pressure is cleanly "
          "separated: median 0.723 versus 0.183. The separation is stable across all three "
          "disjoint windows. Sender flags precede the largest realised 6-month deteriorations "
          "(median +81 patients; +181 in the final window). The structural finding is "
          "geographic. Only 13.6% of the 457 sender months have any in-ICB receiver that "
          "month. 246 of 457 (53.8%) occur in ICBs with fewer than four cohort providers. "
          "There a receiver is arithmetically impossible under the frozen threshold. "
          "In-ICB-only redistribution is starved by design. Cross-ICB or independent-sector "
          "receiving is a policy prerequisite, not a tuning choice. Rule set v1's receiver "
          "condition never fires (0 of 2,337). It is kept and documented as evidence that "
          "band-conditioned receiving is too strict."),
    ("table", ("agent", "Table 8. Frozen agent rules: decision classes over 2,337 "
                        "provider-months (agent_frozen_summary.csv).")),
    ("table", ("stress", "Table 9. v2 stress test: three disjoint 6-month origin windows "
                         "partitioning all 2,337 decisions (eval_c110_agent_stress_h6.csv).")),
    ("fig", ("fig_c110_agent_v2_h6.png",
             "Fig. 11. v2 decision classes (left) and the median realised 6-month change that "
             "followed each class (right), April 2023 to September 2024.", 3.1)),
    ("h2", "5.6 The sealed year adjudicates"),
    ("p", "Table 11 reports the complete one-shot, with each model's validation MAE joined "
          "beside its sealed score. Table 10 and Fig. 12 zoom into the anchor. At C_110, "
          "persistence retakes the lead: 151.5 versus neural pooled 151.6 (n=1,407). The "
          "three-seed-confirmed validation edge did not replicate prospectively. Yet the "
          "broader pattern strengthened. Neural pooled beats persistence at 15 of 23 "
          "specialties. Neural FedAvg and final-cut linear FedAvg win at 14. The exact "
          "validated linear fits win at 10. The largest win is C_330, Dermatology: 163.2 "
          "versus 171.5 (Fig. 13). The dominant effect is the year itself. Per-specialty "
          "persistence MAE drifted by a median +4.6% from validation to sealed year. The "
          "median absolute change is 9.3%, with a range of -51.1% to +38.2% (Fig. 14). The "
          "largest model-over-baseline margin measured anywhere in the project is 4.8% "
          "(C_330)."),
    ("table", ("sealed_anchor", "Table 10. Sealed-year FY 2025-26 MAE at C_110 (n=1,407), with "
                                "each model's validation-year MAE beside it "
                                "(sealed_year_eval.csv).")),
    ("table", ("sealed_full", "Table 11. Sealed-year FY 2025-26: the complete one-shot MAE per "
                              "specialty (sealed_year_eval.csv). Linear and neural models "
                              "trained on the final cut (targets through March 2025); the last "
                              "column is the exact validated linear fit. Bold = best trained "
                              "model where it also beats persistence.")),
    ("fig", ("fig_c110_sealed_one_shot_h1.png",
             "Fig. 12. The one-shot verdict at the anchor: persistence retakes the lead by 0.1 "
             "MAE. Each year is scored on its own rows (validation n=1,401, sealed n=1,407).",
             3.1)),
    ("fig", ("fig_sealed_gap_by_specialty_h1.png",
             "Fig. 13. Sealed-year MAE gap to persistence per specialty, three final-cut models "
             "(below zero favours the model): 14 to 15 of 23 specialties below zero.", 3.1)),
    ("fig", ("fig_sealed_persistence_drift_h1.png",
             "Fig. 14. Persistence MAE change from validation to sealed year, per specialty "
             "(median +4.6%). The year moves more than the models.", 3.1)),
    ("h2", "5.7 Implications"),
    ("p", "Three practical conclusions follow. First, federation is a free privacy layer. In "
          "delta space it costs essentially nothing relative to centralised pooling (Tables 4 "
          "and 5). Data-sharing agreements are not a prerequisite for shared waiting-list "
          "models. Second, point-forecast improvements are not where the value is. "
          "Sub-baseline margins of at most 3.5 MAE did not survive a prospective year, even "
          "when seed-robust (Table 7). The year's own difficulty moved by tens of percent "
          "(Table 11). Calibrated conformal bands (87.6-89.6% coverage) are the honest "
          "planning input. Third, allocation rules fail on geography before they fail on "
          "statistics. The frozen rules separate populations cleanly (Tables 8 and 9). But "
          "half of all sender flags occur where no local receiver can exist. Widening the "
          "receiving pool matters more than refining the score."),

    ("h1", "6. Conclusion and Future Scope"),
    ("p", "fedOPD demonstrates two things on fully public data with a pre-registered "
          "sealed-year protocol. Federated linear and neural clients match centralised "
          "training at zero accuracy cost once models predict changes rather than levels. And "
          "none of them reliably beats naive persistence by more than a rounding error at any "
          "grain or horizon. The deployable system is therefore persistence plus conformal "
          "uncertainty, frozen transparent allocation rules, and federation as the privacy "
          "mechanism. Future work: widening the client pool to the 243 independent-sector "
          "fl_client providers (RTT series without KH03 capacity); cross-ICB and "
          "independent-sector receiving rules for the agent layer; richer exogenous signals "
          "(referral pipelines, workforce); and FL algorithms beyond FedAvg, such as FedProx "
          "[3]. The delta-space result suggests the binding constraint is signal, not "
          "algorithm."),

    ("h1", "References"),
]

REFS = [
    "[1] B. McMahan, E. Moore, D. Ramage, S. Hampson, and B. A. y Arcas, "
    "“Communication-efficient learning of deep networks from decentralized data,” in "
    "Proc. AISTATS, 2017, pp. 1273-1282.",
    "[2] Y. Zhao, M. Li, L. Lai, N. Suda, D. Civin, and V. Chandra, “Federated learning "
    "with non-IID data,” arXiv:1806.00582, 2018.",
    "[3] T. Li, A. K. Sahu, M. Zaheer, M. Sanjabi, A. Talwalkar, and V. Smith, “Federated "
    "optimization in heterogeneous networks,” in Proc. MLSys, 2020.",
    "[4] P. Kairouz et al., “Advances and open problems in federated learning,” "
    "Foundations and Trends in Machine Learning, vol. 14, no. 1-2, 2021.",
    "[5] N. Rieke et al., “The future of digital health with federated learning,” "
    "npj Digital Medicine, vol. 3, no. 119, 2020.",
    "[6] M. J. Sheller et al., “Federated learning in medicine: facilitating "
    "multi-institutional collaborations without sharing patient data,” Scientific "
    "Reports, vol. 10, no. 12598, 2020.",
    "[7] S. Makridakis, E. Spiliotis, and V. Assimakopoulos, “The M4 competition: 100,000 "
    "time series and 61 forecasting methods,” International Journal of Forecasting, "
    "vol. 36, no. 1, pp. 54-74, 2020.",
    "[8] R. J. Hyndman and G. Athanasopoulos, Forecasting: Principles and Practice, 3rd ed. "
    "Melbourne: OTexts, 2021.",
    "[9] D. J. Worthington, “Queueing models for hospital waiting lists,” Journal of "
    "the Operational Research Society, vol. 38, no. 5, pp. 413-422, 1987.",
    "[10] S. A. Kreindler, “Policy strategies to reduce waits for elective care: a "
    "synthesis of international evidence,” British Medical Bulletin, vol. 95, no. 1, "
    "pp. 7-32, 2010.",
    "[11] V. Vovk, A. Gammerman, and G. Shafer, Algorithmic Learning in a Random World. "
    "New York: Springer, 2005.",
    "[12] A. N. Angelopoulos and S. Bates, “Conformal prediction: a gentle "
    "introduction,” Foundations and Trends in Machine Learning, vol. 16, no. 4, "
    "pp. 494-591, 2023.",
    "[13] Y. Romano, E. Patterson, and E. Candès, “Conformalized quantile "
    "regression,” in Proc. NeurIPS, 2019.",
    "[14] K. Cho et al., “Learning phrase representations using RNN encoder-decoder for "
    "statistical machine translation,” in Proc. EMNLP, 2014.",
    "[15] D. P. Kingma and J. Ba, “Adam: a method for stochastic optimization,” in "
    "Proc. ICLR, 2015.",
    "[16] NHS England, “Consultant-led Referral to Treatment waiting times data,” "
    "https://www.england.nhs.uk/statistics/statistical-work-areas/rtt-waiting-times/, "
    "accessed 2026.",
    "[17] NHS England, “Bed availability and occupancy data (overnight, KH03),” "
    "https://www.england.nhs.uk/statistics/statistical-work-areas/"
    "bed-availability-and-occupancy/, accessed 2026.",
]
PLAN += [("ref", r) for r in REFS]
print("plan blocks:", len(PLAN))

plan blocks: 88


In [3]:
# Goal: render the plan into docs/paper/fedopd_paper.docx. Single column, Times New Roman,
# 11 pt body, 8 pt tables with the bold marks, figures centred at their declared widths.

from docx import Document                        # python-docx: writes .docx files directly
from docx.shared import Pt, Inches, RGBColor     # point sizes, image widths, font colours
from docx.enum.text import WD_ALIGN_PARAGRAPH    # paragraph alignment constants

doc = Document()
section = doc.sections[0]
for side in ("top_margin", "bottom_margin", "left_margin", "right_margin"):
    setattr(section, side, Inches(0.8))   # the default 1-inch margins cost the 12-page budget
normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10.5)
normal.paragraph_format.space_after = Pt(4)
for name, size in (("Heading 1", 13), ("Heading 2", 11.5)):
    style = doc.styles[name]
    style.font.name = "Times New Roman"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor(0, 0, 0)   # the default template renders headings blue

CENTER = WD_ALIGN_PARAGRAPH.CENTER


def para(text, size=11, bold=False, align=None, space_after=6):
    # one paragraph with a single run, so font size and weight apply to the whole text
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(size)
    run.font.bold = bold
    if align is not None:
        p.alignment = align
    p.paragraph_format.space_after = Pt(space_after)


def fill_cell(cell, text, bold):
    # table cells come with one empty paragraph; write into it at 8 pt with zero spacing
    p = cell.paragraphs[0]
    p.paragraph_format.space_after = Pt(0)
    run = p.add_run(str(text))
    run.font.size = Pt(8)
    run.font.bold = bold


def render_table(disp, bold, caption):
    para(caption, size=9, space_after=3)
    t = doc.add_table(rows=len(disp) + 1, cols=len(disp.columns))
    t.style = "Table Grid"
    for j, col in enumerate(disp.columns):
        fill_cell(t.rows[0].cells[j], col, bold=True)
    for i, (key, row) in enumerate(disp.iterrows(), start=1):
        for j, col in enumerate(disp.columns):
            marked = bold is not None and col in bold.columns and bool(bold.loc[key, col])
            fill_cell(t.rows[i].cells[j], row[col], bold=marked)
    para("", size=4, space_after=0)   # a hair of air between a table and the next block


def render_fig(png, caption, width):
    doc.add_picture(str(FIGS / png), width=Inches(width))
    doc.paragraphs[-1].alignment = CENTER   # add_picture leaves the image in a fresh paragraph
    para(caption, size=9, align=CENTER)


for line in TITLE:
    para(line, size=16, bold=True, align=CENTER, space_after=2)
for line in AUTHOR:
    para(line, size=10.5, align=CENTER, space_after=2)
for kind, payload in PLAN:
    if kind == "h1":
        doc.add_heading(payload, 1)
    elif kind == "h2":
        doc.add_heading(payload, 2)
    elif kind == "p":
        para(payload)
    elif kind == "ref":
        para(payload, size=9, space_after=2)
    elif kind == "table":
        key, caption = payload
        render_table(*TABLES[key], caption)
    elif kind == "fig":
        render_fig(*payload)

OUT = ROOT / "docs" / "paper" / "fedopd_paper.docx"
doc.save(OUT)
print(f"written {OUT.resolve()}  ({OUT.stat().st_size:,} bytes)")

written C:\Users\heave\Documents\NHS UK FL+Agentic AI\docs\paper\fedopd_paper.docx  (858,032 bytes)


In [4]:
# Goal: verify the stored docx against the csvs. Reopens the saved file so every check runs
# on what is actually on disk, not on the in-memory document.

from docx import Document

saved = Document(OUT)
text = "\n".join(p.text for p in saved.paragraphs)   # prose + headings + captions + references
words = len([w for w in text.split() if any(ch.isalnum() for ch in w)])

assert words <= 4500, words                    # trips if prose grows past the 4,500-word cap
assert len(saved.tables) == 11, len(saved.tables)             # trips if a table dropped out
assert len(saved.inline_shapes) == 14, len(saved.inline_shapes)   # trips if a png went missing
for pin in PINS:
    assert pin in text, pin   # trips if a result csv changed after the prose was written

print(f"fedopd_paper.docx verified: {words:,} words of text (cap 4,500), "
      f"{len(saved.tables)} tables, {len(saved.inline_shapes)} figures, "
      f"all {len(PINS)} headline pins found in the prose")

fedopd_paper.docx verified: 3,582 words of text (cap 4,500), 11 tables, 14 figures, all 11 headline pins found in the prose
